In [12]:
# =========================================================
# silver_validation_testing.ipynb
# Objective:
# Test reusable Silver validation framework using
# bronze_orders dataset.
#
# Project:
# Olist Seller Intelligence Platform
# =========================================================
import sys
import os

# Get project root directory
project_root = os.path.abspath("..")

# Add project root to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root Added:")
print(project_root)

# =========================================================
# 1. CREATE SPARK SESSION
# =========================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SilverValidationTesting") \
    .getOrCreate()

print("Spark Session Created Successfully")



Project Root Added:
/Users/hamid/Desktop/Olist_Seller_Intelligence_Platform
Spark Session Created Successfully


In [13]:

# =========================================================
# 2. LOAD BRONZE ORDERS DATASET
# =========================================================

bronze_orders = spark.read.parquet(
    "../data/bronze/orders/"
)

print("Bronze Orders Dataset Loaded Successfully")


Bronze Orders Dataset Loaded Successfully


In [14]:


# =========================================================
# 3. BASIC DATA INSPECTION
# =========================================================

print("\n==============================")
print("SCHEMA")
print("==============================")

bronze_orders.printSchema()


print("\n==============================")
print("SAMPLE RECORDS")
print("==============================")

bronze_orders.show(5, truncate=False)


print("\n==============================")
print("ROW COUNT")
print("==============================")

print(f"Total Rows: {bronze_orders.count()}")


print("\n==============================")
print("COLUMN NAMES")
print("==============================")

print(bronze_orders.columns)




SCHEMA
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- source_file: string (nullable = true)


SAMPLE RECORDS
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+--------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approve

In [15]:

# =========================================================
# 4. IMPORT VALIDATION FRAMEWORK
# =========================================================

from pipelines.silver.validations.validation_utils import (
    validate_row_count,
    validate_duplicates,
    validate_nulls,
    validate_column_existence,
    validate_numeric_range,
    validate_timestamp_sequence,
    print_validation_result,
    summarize_validations
)

print("Validation Framework Imported Successfully")


Validation Framework Imported Successfully


In [16]:


# =========================================================
# 5. ROW COUNT VALIDATION
# =========================================================

print("\n==============================")
print("ROW COUNT VALIDATION")
print("==============================")

row_count_result = validate_row_count(
    source_df=bronze_orders,
    transformed_df=bronze_orders
)

print_validation_result(row_count_result)



ROW COUNT VALIDATION
VALIDATION: row_count_validation
STATUS: PASSED
MESSAGE: Row count validation passed.
DETAILS:
  - source_count: 99441
  - transformed_count: 99441
  - difference: 0
  - allowed_difference: 0
VALIDATED AT: 2026-05-22T16:12:07.180518


In [17]:


# =========================================================
# 6. DUPLICATE VALIDATION
# =========================================================

"""
Dataset Grain:
ONE ROW = ONE ORDER

Therefore:
order_id must be unique.
"""

print("\n==============================")
print("DUPLICATE VALIDATION")
print("==============================")

duplicate_result = validate_duplicates(
    df=bronze_orders,
    key_columns=["order_id"]
)

print_validation_result(duplicate_result)



DUPLICATE VALIDATION
VALIDATION: duplicate_validation
STATUS: PASSED
MESSAGE: No duplicate records detected.
DETAILS:
  - key_columns: ['order_id']
  - duplicate_group_count: 0
VALIDATED AT: 2026-05-22T16:12:08.014454


In [18]:


# =========================================================
# 7. NULL VALIDATION
# =========================================================

"""
Critical business fields:
- order_id
- order_purchase_timestamp
- order_status
"""

print("\n==============================")
print("NULL VALIDATION")
print("==============================")

null_result = validate_nulls(
    df=bronze_orders,
    critical_columns=[
        "order_id",
        "order_purchase_timestamp",
        "order_status"
    ]
)

print_validation_result(null_result)



NULL VALIDATION
VALIDATION: null_validation
STATUS: PASSED
MESSAGE: No nulls detected in critical columns.
DETAILS:
  - critical_columns: ['order_id', 'order_purchase_timestamp', 'order_status']
  - null_summary: {'order_id': {'null_count': 0, 'null_percentage': 0.0}, 'order_purchase_timestamp': {'null_count': 0, 'null_percentage': 0.0}, 'order_status': {'null_count': 0, 'null_percentage': 0.0}}
VALIDATED AT: 2026-05-22T16:12:08.316200


In [19]:


# =========================================================
# 8. COLUMN EXISTENCE VALIDATION
# =========================================================

print("\n==============================")
print("COLUMN EXISTENCE VALIDATION")
print("==============================")

column_result = validate_column_existence(
    df=bronze_orders,
    expected_columns=[
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

print_validation_result(column_result)




COLUMN EXISTENCE VALIDATION
VALIDATION: column_existence_validation
STATUS: PASSED
MESSAGE: All required columns exist.
DETAILS:
  - expected_columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  - missing_columns: []
VALIDATED AT: 2026-05-22T16:12:08.322130


In [20]:

# =========================================================
# 9. OPTIONAL BUSINESS VALIDATION
# =========================================================

"""
Example:
Validate order status values.

This is exploratory validation only.
"""

print("\n==============================")
print("ORDER STATUS DISTRIBUTION")
print("==============================")

bronze_orders.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)




ORDER STATUS DISTRIBUTION
+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



In [21]:

# =========================================================
# 10. OPTIONAL NULL DISTRIBUTION ANALYSIS
# =========================================================

from pyspark.sql.functions import col, when, count

print("\n==============================")
print("NULL DISTRIBUTION ANALYSIS")
print("==============================")

null_distribution_df = bronze_orders.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in bronze_orders.columns
])

null_distribution_df.show(truncate=False)



NULL DISTRIBUTION ANALYSIS
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+--------------+-----------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|ingestion_timestamp|ingestion_date|source_file|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+--------------+-----------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |0                  |0             |0          |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----

In [22]:


# =========================================================
# 11. VALIDATION SUMMARY
# =========================================================

print("\n==============================")
print("VALIDATION SUMMARY")
print("==============================")

validation_results = [
    row_count_result,
    duplicate_result,
    null_result,
    column_result
]

summary = summarize_validations(validation_results)

print(summary)




VALIDATION SUMMARY
{'overall_status': 'PASSED', 'total_validations': 4, 'passed': 4, 'failed': 0, 'warnings': 0, 'validated_at': '2026-05-22T16:12:08.857173'}


In [23]:

# =========================================================
# 12. OPTIONAL SAVE VALIDATION SUMMARY
# =========================================================

"""
Later:
Save validation reports automatically
inside reports/validation/
"""

# Example future logic:
#
# with open(
#     "reports/validation/orders_validation_report.txt",
#     "w"
# ) as file:
#     file.write(str(summary))


# =========================================================
# 13. STOP SPARK SESSION
# =========================================================

spark.stop()

print("\nSpark Session Stopped Successfully")


# =========================================================
# END OF NOTEBOOK
# =========================================================


Spark Session Stopped Successfully
